Cell 1 — Train three physics-informed models

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from pathlib import Path

sys.path.append(str(Path("../src/models").resolve()))
sys.path.append(str(Path("../src/data").resolve()))
sys.path.append(str(Path("../src/physics").resolve()))
sys.path.append(str(Path("../src/evaluation").resolve()))
sys.path.append(str(Path("../src/training").resolve()))

from train_physics_informed import train_physics_informed
from cnn import HeatSurrogateCNN
from dataset import load_dataset
from metrics import relative_l2, mse

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# train three models with different lambda values
# lambda controls how strongly physics is enforced
configs = [
    {"lambda_energy": 0.1, "lambda_bc": 0.1,
     "lambda_pde": 0.0, "run_name": "pi_lambda_01"},
    {"lambda_energy": 1.0, "lambda_bc": 1.0,
     "lambda_pde": 0.0, "run_name": "pi_lambda_10"},
    {"lambda_energy": 10.0, "lambda_bc": 10.0,
     "lambda_pde": 0.0, "run_name": "pi_lambda_100"},
]

trained_models = {}

for cfg in configs:
    print(f"\n{'='*60}")
    print(f"Training: {cfg['run_name']}")
    print(f"{'='*60}")

    model, history = train_physics_informed(
        dataset_path  = "../data/heat_dataset.npz",
        save_dir      = "../results",
        n_epochs      = 100,
        patience      = 15,
        seed          = 42,
        **cfg,
    )
    trained_models[cfg["run_name"]] = {
        "model"   : model,
        "history" : history,
        "config"  : cfg,
    }

Cell 2 — Compare training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

colors   = ["steelblue", "tomato", "mediumseagreen"]
labels   = ["λ=0.1", "λ=1.0", "λ=10.0"]
run_names = ["pi_lambda_01", "pi_lambda_10", "pi_lambda_100"]

for ax_idx, (run_name, color, label) in enumerate(
    zip(run_names, colors, labels)
):
    h = trained_models[run_name]["history"]

    axes[0].plot(h["train_data"], color=color,
                 label=f"Train {label}", linewidth=2)
    axes[0].plot(h["val_data"],   color=color,
                 label=f"Val {label}",   linewidth=2, linestyle="--")

    axes[1].plot(h["train_energy"], color=color,
                 label=label, linewidth=2)

    axes[2].plot(h["train_bc"], color=color,
                 label=label, linewidth=2)

# also plot baseline for reference
import json
with open("../results/training_history.json") as f:
    baseline_h = json.load(f)

axes[0].plot(baseline_h["val_losses"], color="black",
             linewidth=2, linestyle=":", label="Baseline (λ=0)")

axes[0].set_title("Data Loss (MSE)\nLower = better data fit")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].set_yscale("log")
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Energy Conservation Loss\nLower = less heat retention")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Energy violation")
axes[1].set_yscale("log")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].set_title("Boundary Condition Loss\nLower = better BC enforcement")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("BC violation")
axes[2].set_yscale("log")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle("Physics-Informed Training: Loss Components", fontsize=13)
plt.tight_layout()
plt.savefig("../figures/milestone11_training_curves.png", dpi=150)
plt.show()

Cell 3 — Single-step accuracy comparison

In [ ]:
_, _, test_ds, metadata = load_dataset("../data/heat_dataset.npz")

def evaluate_model_on_test(model, test_ds, device):
    """Compute RelL2 on the full test set."""
    model.eval()
    rel_l2_vals = []

    with torch.no_grad():
        for i in range(len(test_ds)):
            inp, tgt = test_ds[i]
            pred = model(inp.unsqueeze(0).to(device))
            rel_l2_vals.append(
                relative_l2(
                    pred.squeeze().cpu().numpy(),
                    tgt.squeeze().numpy()
                )
            )
    return np.array(rel_l2_vals)

# load baseline
baseline_ckpt = torch.load("../results/best_model.pt",
                            map_location=device)
baseline_model = HeatSurrogateCNN(n_filters=32).to(device)
baseline_model.load_state_dict(baseline_ckpt["model_state"])

results_accuracy = {"Baseline (λ=0)": evaluate_model_on_test(
    baseline_model, test_ds, device)}

for run_name, label in zip(run_names, labels):
    ckpt = torch.load(
        f"../results/{run_name}/best_model.pt",
        map_location=device
    )
    m = HeatSurrogateCNN(n_filters=32).to(device)
    m.load_state_dict(ckpt["model_state"])
    results_accuracy[f"PI CNN ({label})"] = evaluate_model_on_test(
        m, test_ds, device
    )

print("=" * 55)
print("SINGLE-STEP ACCURACY: BASELINE vs PHYSICS-INFORMED")
print("=" * 55)
print(f"{'Model':<22} {'RelL2 Mean':>12} {'RelL2 Std':>12}")
print("-" * 55)
for name, vals in results_accuracy.items():
    print(f"{name:<22} {np.mean(vals)*100:>11.3f}% "
          f"{np.std(vals)*100:>11.3f}%")

Cell 4 — Rollout comparison: the key test

In [ ]:
from heat_solver import make_grid, compute_stable_dt
from boundary_conditions import apply_dirichlet_zero

N      = 64
alpha  = 0.01
stride = 10

x, y, dx = make_grid(N, alpha)
dt, r    = compute_stable_dt(dx, alpha)
X, Y     = np.meshgrid(x, y)

# fixed initial condition for fair comparison
rng  = np.random.default_rng(123)
cx, cy = rng.uniform(0.25, 0.75, size=2)
T0   = np.exp(-((X-cx)**2 + (Y-cy)**2) / (2*0.08**2)).astype(np.float32)
T0   = apply_dirichlet_zero(T0)

n_rollout = 20

def run_cnn_rollout(model, T0, n_steps, device):
    """Autoregressive CNN rollout."""
    snapshots = [T0.copy()]
    T_cur = torch.from_numpy(T0).float().unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(n_steps):
            T_next = model(T_cur)
            T_cur  = T_next
            snapshots.append(T_next.squeeze().cpu().numpy())
    return np.array(snapshots)

def run_solver_rollout(T0, alpha, dx, n_steps, stride):
    from heat_solver import run_simulation
    snaps, _, _, _ = run_simulation(
        T0, alpha, dx,
        n_steps    = n_steps * stride,
        save_every = stride,
    )
    return snaps

# run solver ground truth
true_snaps = run_solver_rollout(T0, alpha, dx, n_rollout, stride)

# run all models
rollout_results = {}
all_models = {"Baseline": baseline_model}
for run_name, label in zip(run_names, labels):
    ckpt = torch.load(f"../results/{run_name}/best_model.pt",
                      map_location=device)
    m = HeatSurrogateCNN(n_filters=32).to(device)
    m.load_state_dict(ckpt["model_state"])
    all_models[f"PI ({label})"] = m

for model_name, m in all_models.items():
    snaps = run_cnn_rollout(m, T0, n_rollout, device)
    rollout_results[model_name] = snaps

Cell 5 — Mean temperature during rollout (the critical comparison)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

steps      = np.arange(n_rollout + 1)
true_means = [true_snaps[s].mean() for s in range(n_rollout + 1)]

colors_r = ["black", "steelblue", "tomato", "mediumseagreen"]

# mean temperature plot
axes[0].plot(steps, true_means, color="black", linewidth=3,
             linestyle="--", label="Solver (ground truth)", zorder=5)

for (model_name, snaps), color in zip(rollout_results.items(), colors_r):
    means = [snaps[s].mean() for s in range(n_rollout + 1)]
    axes[0].plot(steps, means, color=color, linewidth=2,
                 label=model_name, marker="o", markersize=3)

axes[0].set_xlabel("Rollout Step")
axes[0].set_ylabel("Mean Temperature")
axes[0].set_title("Mean Temperature During Rollout\n"
                  "(solver decays — CNN should too)")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# relative L2 error during rollout
axes[1].plot([], [], color="black", linewidth=2,
             label="Solver (reference = 0)")

for (model_name, snaps), color in zip(rollout_results.items(), colors_r):
    rel_l2_curve = [
        relative_l2(snaps[s], true_snaps[s]) * 100
        for s in range(n_rollout + 1)
    ]
    axes[1].plot(steps, rel_l2_curve, color=color,
                 linewidth=2, label=model_name,
                 marker="o", markersize=3)

axes[1].set_xlabel("Rollout Step")
axes[1].set_ylabel("Relative L2 Error (%)")
axes[1].set_title("Rollout Error Accumulation\n"
                  "(lower = better long-term stability)")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle("Rollout Comparison: Baseline vs Physics-Informed CNNs",
             fontsize=13)
plt.tight_layout()
plt.savefig("../figures/milestone11_rollout_comparison.png", dpi=150)
plt.show()

Cell 6 — Fit effective decay rates for all models

In [ ]:
from scipy.optimize import curve_fit

def exp_decay(t, A, lam):
    return A * np.exp(-lam * t)

dt_physical = stride * dt
phys_times  = np.arange(n_rollout + 1) * dt_physical

print("=" * 60)
print("EFFECTIVE DECAY RATE COMPARISON")
print("(solver λ=+0.1147 is the target)")
print("=" * 60)
print(f"{'Model':<22} {'λ (decay rate)':>16} {'Direction':>12}")
print("-" * 60)

# solver
solver_params, _ = curve_fit(exp_decay, phys_times, true_means,
                              p0=[true_means[0], 0.1])
print(f"{'Solver (truth)':<22} {solver_params[1]:>16.4f} "
      f"{'DECAY ✓':>12}")

for (model_name, snaps), color in zip(rollout_results.items(), colors_r):
    means = np.array([snaps[s].mean() for s in range(n_rollout + 1)])
    try:
        params, _ = curve_fit(exp_decay, phys_times, means,
                              p0=[means[0], 0.1])
        direction = "DECAY ✓" if params[1] > 0 else "GROWTH ✗"
        print(f"{model_name:<22} {params[1]:>16.4f} {direction:>12}")
    except Exception:
        print(f"{model_name:<22} {'fit failed':>16}")

print("=" * 60)
print("λ > 0 = physical decay (correct)")
print("λ < 0 = unphysical growth (baseline failure mode)")

Cell 7 — Boundary condition violations during rollout

In [ ]:
print("=" * 55)
print("BOUNDARY CONDITION VIOLATIONS DURING ROLLOUT")
print("(out of 21 steps)")
print("=" * 55)

threshold = 0.01   # same as Milestone 8

solver_bc = sum(
    1 for s in range(n_rollout + 1)
    if (np.abs(true_snaps[s][0,:]).max()  > threshold or
        np.abs(true_snaps[s][-1,:]).max() > threshold or
        np.abs(true_snaps[s][:,0]).max()  > threshold or
        np.abs(true_snaps[s][:,-1]).max() > threshold)
)
print(f"{'Solver (truth)':<22}: {solver_bc:>3} / {n_rollout+1} violations")

for model_name, snaps in rollout_results.items():
    violations = sum(
        1 for s in range(n_rollout + 1)
        if (np.abs(snaps[s][0,:]).max()  > threshold or
            np.abs(snaps[s][-1,:]).max() > threshold or
            np.abs(snaps[s][:,0]).max()  > threshold or
            np.abs(snaps[s][:,-1]).max() > threshold)
    )
    status = "✓" if violations < 5 else "✗"
    print(f"{model_name:<22}: {violations:>3} / {n_rollout+1} "
          f"violations  {status}")

Cell 8 — Final summary table

In [ ]:
print("\n" + "="*70)
print("TABLE 5 — BASELINE vs PHYSICS-INFORMED CNN COMPARISON")
print("="*70)
print(f"{'Model':<22} {'RelL2(%)':>10} {'λ decay':>10} "
      f"{'BC viols':>10} {'Data loss':>12}")
print("-"*70)

for name, vals in results_accuracy.items():
    print(f"{name:<22} {np.mean(vals)*100:>9.3f}%")

print("="*70)
print("\nKey: RelL2 = single-step accuracy (lower=better)")
print("     λ decay = effective decay rate (positive=physical)")
print("     BC viols = boundary violations in 20-step rollout")
print("     Data loss = val MSE (lower=better data fit)")